# RVC → Piper Training Studio — Google Colab

This notebook runs the same headless pipeline as the Windows Studio, but uses a Google Colab NVIDIA GPU.

It can:

- mount Google Drive for persistent datasets/checkpoints;
- load an RVC `.pth` and optional `.index`;
- generate the Piper base dataset and run RVC at **+12** (or any pitch you choose);
- apply the Studio high-pitch audio cleanup before Piper caches the audio;
- warm-start a medium Piper model instead of training the whole VITS/vocoder from scratch;
- resume from `last.ckpt` after a Colab disconnect;
- export the best `val_mel` checkpoint to `.onnx` + `.onnx.json`;
- test the exported Piper voice directly in the notebook.

**Before running:** choose a GPU runtime in Colab (`Runtime` → `Change runtime type` → GPU). GPU availability and session duration are controlled by Colab.


In [ ]:
from google.colab import drive
drive.mount("/content/drive")

import torch
print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("No Colab GPU is active. Change the runtime type to GPU and reconnect.")
print("GPU:", torch.cuda.get_device_name(0))
print("Compute capability:", torch.cuda.get_device_capability(0))


## 1. Clone/update the Studio repo

This notebook always uses the latest `main` branch so the Colab path matches the Windows builder fixes.


In [ ]:
import os, subprocess
REPO = "/content/RVC-to-Piper-Training-APP"

if os.path.isdir(os.path.join(REPO, ".git")):
    subprocess.run(["git", "-C", REPO, "pull", "--ff-only"], check=True)
else:
    subprocess.run([
        "git", "clone",
        "https://github.com/NekoSuneVR/RVC-to-Piper-Training-APP.git",
        REPO,
    ], check=True)

os.chdir(REPO)
print("Repo:", REPO)


## 2. Install the Colab runtime

This creates separate RVC and Piper virtual environments under `/content/rvc-piper-runtime`, builds Piper's native eSpeak/alignment extensions, downloads RMVPE/HuBERT, and downloads the neutral `en_GB-alba-medium` Piper base voice.

The runtime lives on Colab's temporary disk for speed. Your generated dataset, checkpoints, warm-start checkpoint, ONNX and JSON are stored in Google Drive.


In [ ]:
import subprocess
subprocess.run(["bash", "colab/setup_colab.sh", "/content/RVC-to-Piper-Training-APP"], check=True)


## 3. Configuration

Put your RVC model in Google Drive first. A convenient layout is:

```text
MyDrive/RVC-Piper-Colab/models/my-voice.pth
MyDrive/RVC-Piper-Colab/models/my-voice.index
```

The `.index` is optional. Set `rvc_index = ""` if you do not use one.

`generate_dataset = False` lets you reuse an already-created Drive dataset without running RVC again.


In [ ]:
#@title Build settings

voice_name = "en_GB-rvc-custom-medium" #@param {type:"string"}
rvc_model = "/content/drive/MyDrive/RVC-Piper-Colab/models/my-voice.pth" #@param {type:"string"}
rvc_index = "" #@param {type:"string"}

pitch = 12 #@param {type:"integer"}
index_rate = 0.75 #@param {type:"number"}
protect = 0.33 #@param {type:"number"}
f0_method = "rmvpe" #@param ["rmvpe", "pm"]

prompt_file = "/content/RVC-to-Piper-Training-APP/data/piper_training_prompts.txt" #@param {type:"string"}
prompt_limit = 120 #@param {type:"integer"}

batch_size = 8 #@param {type:"integer"}
max_epochs = 1000 #@param {type:"integer"}
checkpoint_every = 5 #@param {type:"integer"}
num_workers = 2 #@param {type:"integer"}

drive_root = "/content/drive/MyDrive/RVC-Piper-Colab" #@param {type:"string"}
generate_dataset = True #@param {type:"boolean"}
resume_if_possible = True #@param {type:"boolean"}

print("Voice:", voice_name)
print("RVC:", rvc_model)
print("Pitch:", pitch)
print("Drive project:", f"{drive_root}/{voice_name}")


## 4. Build everything

This one cell performs the complete pipeline.

For `+12`, the high-pitch cleanup stays enabled. On the first fresh training run it also stores the ~846 MB Piper medium warm-start checkpoint under:

```text
MyDrive/RVC-Piper-Colab/_checkpoints/
```

If Colab disconnects after a checkpoint has been written, reconnect, run the setup cells again, keep `resume_if_possible = True`, and run this cell again. The pipeline will look for the newest `last.ckpt` in Drive and resume it.


In [ ]:
import os, subprocess

RUNTIME = "/content/rvc-piper-runtime"
cmd = [
    "python3",
    "/content/RVC-to-Piper-Training-APP/colab/colab_pipeline.py",
    "--repo-root", "/content/RVC-to-Piper-Training-APP",
    "--rvc-root", f"{RUNTIME}/rvc",
    "--rvc-python", f"{RUNTIME}/rvc-venv/bin/python",
    "--piper-python", f"{RUNTIME}/piper-venv/bin/python",
    "--drive-root", drive_root,
    "--voice-name", voice_name,
    "--rvc-model", rvc_model,
    "--prompts", prompt_file,
    "--prompt-limit", str(prompt_limit),
    "--pitch", str(pitch),
    "--index-rate", str(index_rate),
    "--protect", str(protect),
    "--f0-method", f0_method,
    "--batch-size", str(batch_size),
    "--max-epochs", str(max_epochs),
    "--checkpoint-every", str(checkpoint_every),
    "--num-workers", str(num_workers),
    "--base-model", f"{RUNTIME}/base-voice/en_GB-alba-medium.onnx",
    "--base-config", f"{RUNTIME}/base-voice/en_GB-alba-medium.onnx.json",
]
if rvc_index.strip():
    cmd += ["--rvc-index", rvc_index.strip()]
if not generate_dataset:
    cmd.append("--skip-dataset")
if not resume_if_possible:
    cmd.append("--no-resume")

print("Starting full Colab build...")
subprocess.run(cmd, check=True)


## 5. Check the exported files

Both files stay in Google Drive, so they survive a Colab disconnect.


In [ ]:
from pathlib import Path

project = Path(drive_root) / voice_name / "piper"
onnx_path = project / f"{voice_name}.onnx"
config_path = project / f"{voice_name}.onnx.json"

print("ONNX:", onnx_path, onnx_path.stat().st_size / 1024**2 if onnx_path.exists() else "MISSING", "MB")
print("JSON:", config_path, config_path.exists())

if not onnx_path.exists() or not config_path.exists():
    raise RuntimeError("The final Piper ONNX/JSON pair was not found yet.")


## 6. Test the finished Piper voice

This is **pure Piper inference** — no RVC conversion after synthesis — so it is the correct test for the final standalone model.


In [ ]:
#@title Test text
test_text = "Hello! This is my new standalone Piper voice running from Google Colab." #@param {type:"string"}

import subprocess
from IPython.display import Audio, display
from pathlib import Path

test_wav = Path("/content/test-custom-piper.wav")
piper_python = "/content/rvc-piper-runtime/piper-venv/bin/python"

subprocess.run([
    piper_python,
    "-m", "piper",
    "--model", str(onnx_path),
    "--config", str(config_path),
    "--output-file", str(test_wav),
    "--",
    test_text,
], check=True)

display(Audio(str(test_wav)))
print("Test WAV:", test_wav)


## Notes

- Google Drive contains the persistent project under `<drive_root>/<voice_name>/`.
- Dataset generation is resumable: matching existing WAVs are reused.
- Piper training is resumable from the newest `last.ckpt`.
- In Colab mode, checkpoint writes are intentionally reduced because Google Drive is much slower than local SSD storage.
- The exporter chooses the best `val_mel` checkpoint from the newest training run.
- If a GPU runs out of memory, reduce `batch_size` and rerun. Existing Drive dataset/checkpoints remain available.
